# SALCA Soil Quality (crop level)

## Live demonstration during Brightcon 2026

Mock data with example results displayed for **heavy metals**

Model based on [Oberholzer et al. (2012)](https://doi.org/10.1007/s13593-011-0072-7) and [Nemecek et al. (2024)](https://doi.org/10.1007/s11367-023-02255-w)

![](https://media.springernature.com/full/springer-static/image/art%3A10.1007%2Fs13593-011-0072-7/MediaObjects/13593_2011_72_Fig1_HTML.gif?as=webp)

Figure source: [Oberholzer et al. (2012)](https://doi.org/10.1007/s13593-011-0072-7)

![](SALCAsoilquality_steps_crop.png)

In [ ]:
"""
File name: SALCAsoilquality_crop.ipynb
Created on Tue May 14 15:22:38 2019

Author: Agroscope, 'Life Cycle Assessment' Research Group, Switzerland
Date: 2026-09-16
Version: 1.0.1-demo
Licence: GNU LGPLv3

Description: 
    This script assesses the impact of agricultural management practices on
    soil quality.

Calculation level:
    Crop

Related scripts: SALCAheavymetal_crop.py, SALCAmapping_ppp.py, SALCAanimal_farm.py
    It uses input from these scripts.
"""

In [ ]:
# %% import modules ===========================================================

import json as js
import re
import warnings as w
from sys import stderr
from dataclasses import dataclass, field
from collections import namedtuple

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# from salcaconnector import create_salca_context  # only relevant to Agroscope-internal use

In [ ]:
# %% start sequence ===========================================================

class SALCAconnectorContext:
    def __init__(self, d = None):
        if d is not None:
            for key, value in d.items():
                setattr(self, key, value)

class SALCAconnectorInput:
    def __init__(self, d = None):
        if d is not None:
            for key, value in d.items():
                setattr(self, key, value)

class SALCAconnectorOutput:
    pass

def dict2tuple(d):
    return namedtuple('Crop', d.keys())(**d)

with open('SALCAsoilquality_ctx.json', mode = 'r') as f:
    ctx = SALCAconnectorContext(js.load(f))

ctx.crop = dict2tuple(ctx.crop)

with open('SALCAsoilquality_crop_data.json', mode = 'r') as f:
    inp = SALCAconnectorInput(js.load(f))

out = SALCAconnectorOutput()

In [ ]:
# %% define input data classes ================================================

@dataclass
class FarmData:
    """Data class storing the farm-level input data."""

    climate_type: str
    climate_zone: str
    sman_kgospt: float
    lman_kgospm3: float

@dataclass
class FieldData:
    """Data class storing the field-level input data."""

    soil_type: str
    soil_kgpm3: float
    soil_depth_cm: float
    clay_frac: float
    humus_perc: float
    ph: float

@dataclass
class CropData:
    """Data class storing the crop-level input data."""

    crop: str
    ca_ha: float
    protective_sowi: int
    eros_permaculture: int

@dataclass
class GrazingData:
    """Data class storing the input data about grazing."""

    no_days: list
    no_lsu: list
    livestock_type: list
    h: list = field(init=False)

    def __post_init__(self):
        """Extract daily grazing hours based on the animal category."""
        self.h = [extract_past_h(val) for val in self.livestock_type]

@dataclass
class EventData:
    """Data class storing the input data about events."""

    harv: dict
    harv_sepr: dict
    sowi: dict
    maint: dict
    ppp: dict
    fertilisation: dict
    manure_l: dict
    manure_s: dict
    soil_trea: dict

@dataclass
class HeavyMetalData:
    """Data class storing the input data about heavy metals."""

    soil_cd_mgpa: float
    soil_cu_mgpa: float
    soil_zn_mgpa: float
    soil_pb_mgpa: float
    soil_ni_mgpa: float
    soil_cr_mgpa: float
    soil_hg_mgpa: float

@dataclass
class OrganicPollutantData:
    """Data class storing the input data about organic pollutants."""

    soil_pcdd_pcdf: float
    soil_pah: float
    soil_pcb: float

@dataclass
class PrepSQ:
    """Data class collecting the input data needed in this module."""

    f: FarmData
    fi: FieldData
    c: CropData
    in_past: GrazingData
    event: EventData
    hm: HeavyMetalData
    op: OrganicPollutantData
    mc_dict: list[dict]

In [ ]:
# %% define functions for input data and data processing ======================

def extract_past_h(val):
    """Extract daily grazing hours based on the animal category."""
    f_salcaprep = js.loads(inp.f_salcaprep)
    try:
        return f_salcaprep['f_past_h'][val]
    except KeyError as e:
        w.warn((f"KeyError with key {e}\n"
                f"Animal category {val} does not exist on the farm. "
                f"A standard grazing duration of 4 hours is assumed for this animal category."))
        stderr.flush()  # show warning immediately
        return 4


def read_js_table(table):
    """Convert a JSON string to a data frame."""
    return pd.DataFrame(data=js.loads(table)['data'],
                        columns=js.loads(table)['columns'],
                        index=js.loads(table)['index'])


def get_from_table(tab, col, col2, name):
    """
    Derive a value from a table.

    Keyword arguments:
    tab: Table to extract values from.
    col: Column in the table to extract values from.
    col2: Column in the table with the condition of the row.
    name: Match in col2 to derive the row of the extracted value.
    """
    try:
        value = tab[col][tab[col2] == name].values[0]
    except ValueError as exc:
        raise ValueError(f"Value could not be found in the table for "
                         f"'{name}' in column '{col}'") from exc
    return value


def estimate_agricultural_threshold(threshold, is_agri):
    """
    Estimate the pollution thresholds under specific environmental conditions
    for either cropland or pasture.
    """
    sampling_depth_m = 0.2  # sampling depth [m]

    return prep_sq.c.ca_ha \
        * factors['ha2m2'] \
            * sampling_depth_m \
                * prep_sq.fi.soil_kgpm3 \
                    * threshold \
                        * is_agri

In [ ]:
# %% define functions for impact classes ======================================

def evaluate_wheeling():
    """Evaluate risk of soil compaction by wheeling."""
    return None


def evaluate_grazing():
    """Evaluate risk of soil compaction by grazing."""
    return None


def evaluate_structure_stabilisation(crop, is_grass, is_arable, tab_crops):
    """
    Evaluate stabilisation of soil structure by plants.
    This is also relevant to positive effects of plants on earthworm populations.
    """
    cultural_value = get_from_table(tab_crops, 'SQ Cultural Value', 'NameEN', crop)

    # cultural value adding to the crop rotation value of grassland and arable crops
    crop_rotation_grass = cultural_value * is_grass
    crop_rotation_arable = cultural_value * is_arable

    return crop_rotation_grass, crop_rotation_arable


def evaluate_structure_formation(crop, hres_present, tab_crops):
    """Evaluate structure build-up by straw amendment."""
    if get_from_table(tab_crops, 'SQ Crop Residues soil texture formation', 'NameEN', crop) == 1:
        soil_texture_hres_frac = hres_present
    else:
        soil_texture_hres_frac = 0

    return soil_texture_hres_frac


def evaluate_humus_balance():
    """Evaluate humus dynamics."""
    return None


def evaluate_earthworm_protection_cutting(harv_event, is_grass):
    """Evaluate positive effects of high cutting levels on earthworm populations."""
    # cutting level of grassland
    worm_protection_cutting_level = 0
    for idx in np.arange(len(harv_event['cutting_level'])):
        if harv_event['cutting_level'][idx] == '002 >8cm':
            worm_protection_cutting_level += is_grass * \
                1/len(harv_event['cutting_level'])  # weighted by number of harvest events

    return worm_protection_cutting_level


def evaluate_earthworm_impact_tillage():
    """Evaluate potential impact of soil tillage on earthworms."""
    return None


def evaluate_earthworm_impact_seedbed(protective_sowi):
    """Evaluate potential impact of seedbed preparation on earthworms."""
    # protective sowing is evaluated in SALCAprep_crop
    return protective_sowi


def evaluate_pollutants(*, pollutant, data, thresholds, is_grass, is_arable, counter):
    """
    Evaluate input of heavy metals and organic pollutants.
    This is done by estimating the time until reaching the pollution thresholds.
    """
    if counter == 0:
        pollutant_type = re.sub(r'(?<!^)(?=[A-Z])', ' ',
                                type(data).__name__.replace('Data','')) + 's'

    threshold = thresholds[pollutant.replace('_mgpa', '')]
    threshold_arable = estimate_agricultural_threshold(threshold, is_arable)
    threshold_grass = estimate_agricultural_threshold(threshold, is_grass)

    # time until the threshold value is reached with the current input rate [a]
    # if there are no pollutants, a default number of years is returned
    # to avoid a division by zero
    default_a = 30000  # > 300 y -> evaluation will be 0

    pollutant_input = getattr(data, 'soil_' + pollutant)
    if pollutant_input > 0:
        time_to_threshold_arable_a = threshold_arable / pollutant_input
        time_to_threshold_grass_a = threshold_grass / pollutant_input
    else:
        time_to_threshold_arable_a = default_a
        time_to_threshold_grass_a = default_a

    if type(data).__name__ == 'HeavyMetalData':
        pollutant = pollutant.replace('_mgpa', '')
        pollutant = pollutant[0].upper() + pollutant[1]
    elif type(data).__name__ == 'HeavyMetalData':
        pollutant = pollutant.upper().replace('_','/')

    return time_to_threshold_grass_a, time_to_threshold_arable_a


def evaluate_slurry_application(n_manure_l, is_grass, is_arable):
    """Evaluate toxic effects of slurry application."""
    evaluation_slurry_grassland = 0
    evaluation_slurry_arable = 0

    # If more than three slurry events (grassland)
    if n_manure_l > 3:
        evaluation_slurry_grassland = is_grass
    # If more than one slurry event (arable areas)
    if n_manure_l > 1:
        evaluation_slurry_arable = is_arable

    return int(evaluation_slurry_grassland), int(evaluation_slurry_arable)


def evaluate_organic_substances():
    """Evaluate input of stable and rapidly degradable organic fertiliser."""
    return None


def evaluate_liming(ph, fertilisation_event, ca_ha, is_arable, is_grass):
    """Evaluate liming at pH < 6.2."""
    # derive if the applied lime [kg] exceeds 10 dt/ha (= 1000 kg/ha)
    lime_kg = 0
    for idx, fert in enumerate(fertilisation_event['variable']):
        if fert in ['Ca Kalk', 'Ca Karbonationskalk', 'Ca Meeresalgenkalk']:
            lime_kg += fertilisation_event['lime_kg'][idx]
    if lime_kg*factors['kg2dt'] / ca_ha > 10:
        lime_10dt = 1
    else:
        lime_10dt = 0

    if ph < 6.2:
        ph_lime_arable = is_arable * lime_10dt
        ph_lime_grass = is_grass * lime_10dt
    else:
        ph_lime_arable = 0
        ph_lime_grass = 0

    return int(ph_lime_arable), int(ph_lime_grass)


def evaluate_ppp_application(n_ppp, is_grass, is_arable):
    """Evaluate toxic effects of pesticide application."""
    evaluation_ppp_grassland = 0
    evaluation_ppp_arable = 0

    # more than zero ppp events (grassland or arable areas)
    if n_ppp > 0:
        evaluation_ppp_grassland = is_grass
        evaluation_ppp_arable = is_arable

    return int(evaluation_ppp_grassland), int(evaluation_ppp_arable)

In [ ]:
# %% load input data ==========================================================

c_salcaprep_str = inp.c_salcaprep
c_salcaprep = js.loads(c_salcaprep_str)

prep_sq = PrepSQ(

    f = FarmData(
        climate_type = inp.f_climate_type,  # Climate type (arid, humid, wet)
        climate_zone = inp.f_climate_zone[4:],  # Climate zone (A - G)

        # organic substance (OS) content in farm manure (computed in SALCAanimal)
        sman_kgospt = inp.f_sman_kgOSpt,
        lman_kgospm3 = inp.f_lman_kgOSpm3),

    fi = FieldData(
        soil_type = inp.fi_soil_type,  # soil type (sand, loam, silt, clay)
        soil_kgpm3 = inp.fi_soil_kgpm3,  # soil density [kg/m3]
        clay_frac = inp.fi_clay_frac,  # clay content of the soil [0,1], from SALCAprep_field
        humus_perc = inp.fi_humus_perc,  # humus content of the soil [%]
        ph = inp.fi_pH,  # pH of the soil []
        soil_depth_cm = inp.fi_soil_depth_cm),  # soil depth [cm]

    # derive crop type from context (string, Python code of the crop)
    c = CropData(
        crop = ctx.crop.crop_definition_name,
        ca_ha = inp.c_ca_ha,  # crop area (cropland or pasture) [ha]

        # derive information from SALCAprep_crop
        # protective sowing applies for crop?
        protective_sowi = c_salcaprep['c_protective_sowi'],
        # is the crop a permanent crop or older grassland ley?
        eros_permaculture = c_salcaprep['c_eros_permaculture']),  # for soil looseness

    # grazing takes place at the crop level
    in_past = GrazingData(
        no_days = [value['c_past_d'] for value in inp.c_grazing],  # days
        no_lsu = [value['c_past_LU'] for value in inp.c_grazing],  # livestock units
        livestock_type = [value['c_past_LU_type']
                          for value in inp.c_grazing]),  # livestock type

    # heavy metal inputs into the soil on crop level, from SALCAheavymetal [mg/a]
    hm = HeavyMetalData(
        soil_cu_mgpa = inp.c_e_soil_cu_mg,
        soil_cd_mgpa = inp.c_e_soil_cd_mg,
        soil_zn_mgpa = inp.c_e_soil_zn_mg,
        soil_pb_mgpa = inp.c_e_soil_pb_mg,
        soil_ni_mgpa = inp.c_e_soil_ni_mg,
        soil_cr_mgpa = inp.c_e_soil_cr_mg,
        soil_hg_mgpa = inp.c_e_soil_hg_mg),

    # organic pollutant inputs
    op = OrganicPollutantData(
        soil_pcdd_pcdf = np.nan,  # required input data not available
        soil_pah = np.nan,  # required input data not available
        soil_pcb = np.nan),  # required input data not available

    # derive information from SALCAprep_crop
    event = EventData(
        harv = c_salcaprep['tr_harv'],  # harvest
        harv_sepr = c_salcaprep['tr_harv_sepr'],  # harvest secondary product (e.g. straw)
        sowi = c_salcaprep['tr_sowi'],  # sowing
        maint = c_salcaprep['tr_maint'],  # maintenance events
        ppp = c_salcaprep['tr_ppp'],  # plant protection products
        fertilisation = c_salcaprep['tr_fert'],  # mineral fertilisation
        manure_l = c_salcaprep['tr_manure_l'],  # liquid manure
        manure_s = c_salcaprep['tr_manure_s'],  # solid manure
        soil_trea = c_salcaprep['tr_soil']),  # soil treatment

    mc_dict = inp.treatment_maco_event  # machine combinations
)

In [ ]:
# %% load reference tables ====================================================

factors = {'kg2dt': 1/100,  # convert kg to dt
           'ha2m2': 10000,  # convert ha to m2
           'density_slurry_kgpm3': 1000,  # slurry density [kg/m3]
           't2kg': 1000}  # convert tonnes to kg

# load crop table
tab_crops = read_js_table(inp.tab_crops)

# threshold values for heavy metal content [mg/kgTS] of soil
hm_thresholds_mgpkg = {'cd': 0.8,
                       'cu': 40,
                       'zn': 150,
                       'pb': 50,
                       'ni': 50,
                       'cr': 50,
                       'hg': 0.5}

# threshold values for organic pollutant content of soil
op_thresholds = {'pcdd_pcdf': 5,  # [ng toxic equivalents (TEQ) kg-1 DM]
                 'pah': 1,  # [mg kg-1 DM]
                 'pcb': 0.2}  # [mg kg-1 DM]

In [ ]:
# %% evaluate impact classes ==================================================

hres_present = prep_sq.event.harv_sepr['date'] == []

# dict to collect the results from the impact class evaluation
sq_crop = {}

# derive if the crop is grassland or arable land
sq_crop['c_is_grass'] = int(prep_sq.c.eros_permaculture)
sq_crop['c_is_arable'] = int(1-prep_sq.c.eros_permaculture)

# erosion only evaluated at the farm level

sq_crop['c_tr_evaluation'] = 0  # evaluate_wheeling()

sq_crop['c_graz_evaluation'] = 0  # evaluate_grazing()

sq_crop['c_crop_rotation_grass'], sq_crop['c_crop_rotation_arable'] = \
    evaluate_structure_stabilisation(prep_sq.c.crop, sq_crop['c_is_grass'],
                                     sq_crop['c_is_arable'], tab_crops)

sq_crop['c_soil_texture_hres_frac'] = \
    evaluate_structure_formation(prep_sq.c.crop, hres_present, tab_crops)

sq_crop['c_humus_balance_net_kg'], \
    sq_crop['c_humus_balance_gross_kgpha'], \
        sq_crop['c_humus_balance_area_ha'] = [-445.0, -426.0, 1.0]  # evaluate_humus_balance()

# positive effects of plants on earthworm populations evaluated through
# output of evaluate_structure_stabilisation

sq_crop['c_worm_protection_cutting_level']  = \
    evaluate_earthworm_protection_cutting(prep_sq.event.harv, sq_crop['c_is_grass'])

sq_crop['c_ew_phys_sum'] = 3.18  # evaluate_earthworm_impact_tillage()

sq_crop['c_protective_sowi'] = evaluate_earthworm_impact_seedbed(prep_sq.c.protective_sowi)

for i, hm in enumerate([hm.replace('soil_', '') for hm in list(vars(prep_sq.hm))]):
    sq_crop['c_ttt_grass_' + hm.replace('_mgpa', '') + '_a'], \
    sq_crop['c_ttt_arable_' + hm.replace('_mgpa', '') + '_a'] = \
            evaluate_pollutants(pollutant = hm, data = prep_sq.hm,
                                thresholds = hm_thresholds_mgpkg,
                                is_grass = sq_crop['c_is_grass'],
                                is_arable = sq_crop['c_is_arable'], counter = i)

for i, op in enumerate([op.replace('soil_', '') for op in list(vars(prep_sq.op))]):
    sq_crop['c_ttt_grass_' + op + '_a'], sq_crop['c_ttt_arable_' + op + '_a'] = \
            evaluate_pollutants(pollutant = op, data = prep_sq.op,
                                thresholds = op_thresholds,
                                is_grass = sq_crop['c_is_grass'],
                                is_arable = sq_crop['c_is_arable'], counter = i)

sq_crop['c_evaluation_slurry_grassland'], sq_crop['c_evaluation_slurry_arable'] = \
    evaluate_slurry_application(len(prep_sq.event.manure_l['amount_m3']),
                                sq_crop['c_is_grass'], sq_crop['c_is_arable'])

sq_crop['c_org_subst_stable_kgpha'], sq_crop['c_org_subst_degradable_kgpha'] = \
    [243.0, 347.0]  # evaluate_organic_substances()

sq_crop['c_pH_lime_arable'], sq_crop['c_pH_lime_grass'] = \
    evaluate_liming(prep_sq.fi.ph, prep_sq.event.fertilisation, prep_sq.c.ca_ha,
                    sq_crop['c_is_arable'], sq_crop['c_is_grass'])

sq_crop['c_evaluation_ppp_grassland'], sq_crop['c_evaluation_ppp_arable'] = \
    evaluate_ppp_application(len(prep_sq.event.ppp['date']),
                             sq_crop['c_is_grass'], sq_crop['c_is_arable'])

In [ ]:
# %% export output ============================================================

out.c_sq_output = js.dumps(sq_crop)

In [ ]:
# %% example results for heavy metals =========================================

# subset of results for heavy metals on arable land
regex = re.compile('c_ttt_arable_.{2}_a')
hm_keys = list(filter(regex.match, sq_crop.keys()))
hm_results = pd.Series({key: sq_crop[key] for key in hm_keys})
hm_results = hm_results.sort_index(ascending = False)

# highlight of most critical heavy metal
my_color = np.where(hm_results==hm_results.min(), '#E60005', '#909CC2')
my_size = np.where(hm_results==hm_results.min(), 70, 30)
my_text = str(round(min(hm_results)))

# lollipop chart
plot_range = range(1,len(hm_results)+1)
plt.hlines(y = plot_range, xmin = 0, xmax = hm_results.values, color = my_color)
plt.scatter(hm_results.values, plot_range, color = my_color, s = my_size)
plt.ylabel('Heavy metal')
plt.yticks(plot_range, [label.replace('c_ttt_arable_', '').replace('_a', '') for label in hm_results.index])
plt.xlabel('Time to threshold (years)')
plt.xlim(left = 0)
plt.annotate(my_text, (hm_results.min() + 1000, np.argmin(hm_results) + 0.95))
plt.show()